In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
from typing import List, Dict
import os
from urllib.parse import urljoin
from pprint import pprint
import re

BASE_PATH = "../../data/RAG"

## 1. URL 엔드포인트 분석
- **대상 URL**: `https://maplestory.nexon.com/Guide/N23GameInformation/Articles/272`
- **HTTP 메서드**: GET
- **설명**: 메이플스토리 게임 정보 가이드라인
    - 카테고리 : 기초 가이드, 성장, 아이템, 사냥/보스 콘텐츠, 스페셜 콘텐츠, 커뮤니티, 거래, 캐시 & 코디, 기타/TIP

### User-Agent 설정
서버에 '일반 사이트에서 접근하고 있다'고 알려주는 것
```
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0"}
```
개발자 도구에서 확인할 수 있음

### API 문서 읽는 법
- **엔드포인트** : 요청을 보내는 API의 주소
- **메서드** : 가져오기 - GET, 보내기 - POST
- **요청 헤더** : 인증키를 담는 곳
- **필수 파라미터** : 요청에 함께 보내는 조건(키=값)
    - 안넣으면 400에러
- 응답 필드 : 내가 꺼내 쓸 키 이름 미리 확인하기

#### 요청 보내기
```
response = requests.get(url, params=params)
```
- requests 라이브러리로 get 함수 사용하여 요청 보내기
- 인자값으로 url, header, params(필요하다면),timeout

In [ ]:
def get_maple_guide(url: str) -> str:
    """메이플 가이드라인 문서 크롤링

    Args:
        page (int): 페이지 번호

    Returns:
        str: 웹페이지의 HTML 내용
    """
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0"}

    try:
        response = requests.get(url, headers=headers, timeout=20)
        response.raise_for_status()  # 오류가 있으면 예외를 발생시킴
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"페이지 요청 중 에러 발생: {e}")
        return ""

## 2. 가이드문서 게시글 아이디 가져오기 
*함수 사용X(boardId는 수작업으로 뽑아서 딕셔너리로 만듦)*
### Request URL 확인 
- 기초 가이드의 Request URL : https://maplestory.nexon.com/guide/n23gameinformation/articles?boardId=429467337
- 성장의 Reqeust URL : https://maplestory.nexon.com/guide/n23gameinformation/articles?boardId=429467338

    - 가이드문서 엔드포인트 url을 확인해보면 파라미터로 boardId를 사용하고 있음

- 가이드의 내용은 카테고리 안의 있고 그 카테고리의 boardId를 가져오고 있음

    - 가져오고 있기 때문에 `post` 방식 사용

1. 파라미터 설정 : `params = {"boardId" : board_id}`
2. 헤더 설정 : `headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0"}`
3. POST 방식으로 가져오기 : `response = requests.post(url, params=params, headers=headers, timeout=20)`

In [3]:
def get_guide_list(board_id: int):
    url = "https://maplestory.nexon.com/guide/n23gameinformation/articles"

    params = {
        "boardId": board_id
    }

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "Chrome/120.0.0.0"
        )
    }

    response = requests.post(
        url,
        params=params,
        headers=headers,
        timeout=20
    )

    response.raise_for_status()

    return response.json()

## 3. 카테고리의 세부 내용 가져오기
### 개발자도구에서 Response 확인
```JSON
    "pageInfo": {
        "numberOfElements": 17,
        "totalElements": 17,
        "totalPages": 1
    },
    "list": [
        {
            "boardId": 429467338,
            "boardName": null,
            "articleId": 377,
            "categoryId": null,
            "subCategoryId": null,
            "title": "[캐릭터] 레벨",
            "content": null,
            "textContent": "■ 목차\n1. 레벨업 방법\n2. 레벨업 시 획득 가능한 포인트\n3. 레벨별 데미지 공식\n4. 레벨별 경험치 획득량\n\n\n\n\n1. 레벨업 방법\n\n■ 사냥, 퀘스트, 컨텐츠 참여 등을 통해 경험치를 얻어 레벨을 올릴 수 있음\n\n■ 길라잡이(기본 단축키 [U])를 통해, 레벨대에 레벨대에 맞는 사냥터 및 퀘스트를 확인할 수 있음\n\n\n\nTIP\n카데나, 아크, 일리움, 제로는 초반 튜토리얼을 진행하지 않으면 스토리 퀘스트 외의 지역에서 경험치 획득 불가능\n이벤트 보상이나 컨텐츠 코인샵을 통해 획득 가능한 ‘비약’ 아이템을 통해서도 경험치 획득이 가능\n빠른 레벨업을 위해선? [알아보러 가기]\n\n\n\n\n\n\n\n2. 레벨업 시 획득 가능한 포인트\n\n1) AP [자세히 알아보기]\n\n■ 힘(STR), 민첩(DEX), 지력(INT), 운(LUK)의 능력치와 HP, MP의 양을 올릴 수 있는 어빌리티 포인트 (기본 단축키 [S])\n\n\n\n\n\n2) SP [자세히 알아보기]\n\n■ 스킬의 레벨을 올릴 수 있는 스킬 포인트 (기본 단축키 : [K])\n\n \n\n\n\n\n3) 하이퍼 스탯/스킬\n■ 하이퍼 스탯 [자세히 알아보기]\n\n\n\n - 하이퍼 스탯 포인트를 사용하여 원하는 스탯 레벨을 올릴 수 있음\n\n\n\n\n\n\n\n\n■ 하이퍼 스킬 [자세히 알아보기]\n\n- 140레벨을 달성하면 익힐 수 있는 스킬\n\n- 패시브와 공격/버프 액티브 스킬을 배울 수 있음\n\n\n\n\n 4) V매트릭스 포인트 [자세히 알아보기]\n\n■ 200레벨 이후 레벨업을 할 때마다 V 매트릭스 포인트 1P 획득 가능\n\n■ 매트릭스 1P 당 1단계의 슬롯 강화가 가능하며, 슬롯 강화 단계만큼 코어 레벨 상승 \n\n\n\n\n\n3. 레벨별 데미지 공식\n몬스터와의 레벨 차이에 따른 데미지\n\n\n캐릭터 레벨 - 몬스터 레벨\n\n\t\n\n데미지\n\n\t\n\n비고\n\n\n\n\nLv 5 이상\n\n\t\n\n120%\n\n\t\n\n\n\n\n\n\nLv 4 ~ 1\n\n\t\n\n118% ~ 110%\n\n\t\n\n1레벨당 2%p씩 감소\n\n\n\n\n0\n\n\t\n\n110%\n\n\t\n\n\n\n\n\n\nLv -1\n\n\t\n\n105.3%\n\n\t\n\n\n\n\n\n\nLv -2\n\n\t\n\n100.7%\n\n\t\n\n\n\n\n\n\nLv -3\n\n\t\n\n96.2%\n\n\t\n\n\n\n\n\n\nLv -4\n\n\t\n\n91.8%\n\n\t\n\n\n\n\n\n\nLv -5~-40 이상\n\n\t\n\n87.5% ~ 0%\n\n\t\n\n1레벨 당 2.5%p씩 감소\n\n\n\n\n\n4. 레벨별 사냥 경험치 획득량\n몬스터와의 레벨 차이에 따른 경험치 획득량\n\n\n캐릭터 레벨 – 몬스터 레벨\n\n\t\n\n경험치 획득량\n\n\t\n\n비고\n\n\n\n\nLv 40 이상\n\n\t\n\n70%\n\n\t\n\n\n\n\n\n\nLv 39~21\n\n\t\n\n71 ~ 89%\n\n\t\n\n1레벨 당 1%p씩 증가\n\n\n\n\nLv 20~19\n\n\t\n\n95%\n\n\t\n\n\n\n\n\n\nLv 18~17\n\n\t\n\n96%\n\n\t\n\n\n\n\n\n\nLv 16~15\n\n\t\n\n97%\n\n\t\n\n\n\n\n\n\nLv 14~13\n\n\t\n\n98%\n\n\t\n\n\n\n\n\n\nLv 12~11\n\n\t\n\n99%\n\n\t\n\n\n\n\n\n\nLv 10\n\n\t\n\n100%\n\n\t\n\n\n\n\n\n\nLv 9~5\n\n\t\n\n105%\n\n\t\n\n\n\n\n\n\nLv 4~2\n\n\t\n\n110%\n\n\t\n\n\n\n\n\n\nLv 1~-1\n\n\t\n\n120%\n\n\t\n\n\n\n\n\n\nLv -2~-4\n\n\t\n\n110%\n\n\t\n\n\n\n\n\n\nLv -5~-9\n\n\t\n\n105%\n\n\t\n\n\n\n\n\n\nLv -10~-20\n\n\t\n\n100 ~ 90%\n\n\t\n\n1레벨 당 1%p씩 감소\n\n\n\n\nLv -21~-35\n\n\t\n\n70 ~ 14%\n\n\t\n\n1레벨 당 4%p씩 감소\n\n\n\n\nLv -36~-39\n\n\t\n\n10%\n\n\t\n\n\n\n\n\n\nLv -40 이하\n\n\t\n\n최대 100",
            "createDate": "\/Date(1748209928000)\/",
            "isNew": false
        },
```
- 세부내용의 고유 `articleId`가 존재함
- RAG 문서를 위해 `articleId`, `title`, `textContent`, `url` 가져오기

In [ ]:
def parse_api_guide(item: dict) -> dict:

    article_id = item["articleId"]

    return {
        "article_id": article_id,
        "title": item["title"],
        "content": item["textContent"],
        "url": (
            "https://maplestory.nexon.com/"
            f"Guide/N23GameInformation/Articles/{article_id}"
        )
    }

In [ ]:
board_ids = {
    "기초 가이드": 429467337,
    "성장": 429467338,
    "아이템": 429467339,
    "사냥/보스 컨텐츠": 429467340,
    "스페셜 컨텐츠": 429467341,
    "커뮤니티": 429467342,
    "거래": 429467343,
    "캐시 & 코디": 429467344,
    "기타/TIP": 429467345,
}

# 수집한 문서를 담을 리스트 선언
all_guides = []

# board_ids 딕셔너리를 순회
# category: 기초 가이드, 성장, 아이템 등 - key
# board_id: 429467337, 429467338 등 - value
for category, board_id in board_ids.items():
    # 가이드문서의 본문을 뽑기 위해 딕셔너리로 선언한 board_id를 인자로 넣음
    data = get_guide_list(board_id)
    # 위에서 확인한 개발자도구의 Response의 list의 키(key) 가져오기
    for item in data['list']:
        # parse_api_guide 함수에 key를 넣어서 guide 변수에 저장
        guide = parse_api_guide(item)
        # 카테고리 이름과 카테고리 아이디도 함께 문서에 저장
        guide['category'] = category
        guide['board_id'] = board_id
        # 모두 문서에 저장
        all_guides.append(guide)

print("수집 문서 개수:", len(all_guides))


수집 문서 개수: 106


In [22]:
def save_guides_to_json(guides, BASE_PATH, file_name="maple_guides.json"):

    file_path = os.path.join(BASE_PATH, file_name)

    with open(file_path, "w", encoding='utf-8') as f:
        json.dump(guides, f, ensure_ascii=False, indent=2)

    print("JSON 저장 완료")
    return file_path

save_guides_to_json(all_guides, BASE_PATH)

JSON 저장 완료


'../../data/RAG\\maple_guides.json'